## 📊 Évaluation d'un modèle CNN sur des images de radiographies pulmonaires

### 📁 Chargement des bibliothèques et configuration des chemins
Importation des bibliothèques nécessaires et configuration des répertoires pour l'évaluation du modèle.

In [ ]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
import json
from datetime import datetime
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report
import matplotlib.pyplot as plt

### 📂 Sélection et Chargement Dynamique du Modèle
Ce bloc de code permet de parcourir un répertoire spécifié pour identifier les modèles sauvegardés au format .h5, puis de les lister pour que l'utilisateur puisse en sélectionner un. Une fois le choix effectué, le modèle est chargé en mémoire pour être utilisé dans des tâches d'évaluation ou de prédiction.

In [ ]:
# Définition du répertoire contenant les modèles sauvegardés
model_dir = os.path.join(os.getcwd(), 'models')
model_files = [f for f in os.listdir(model_dir) if f.endswith(".h5")]

# Vérification de la présence de modèles disponibles
if not model_files:
    print("Aucun modèle trouvé dans le dossier 'models'.")
    exit()

# Affichage de la liste des modèles disponibles
print("Modèles disponibles :")
for idx, m in enumerate(model_files):
    print(f"{idx} : {m}")

# Demande de sélection du modèle à charger
choice = input("Choisissez le numéro du modèle à utiliser : ")

# Validation de l'entrée utilisateur
try:
    choice_index = int(choice)
    chosen_model_file = model_files[choice_index]
except (ValueError, IndexError):
    print("Choix invalide.")
    exit()

# Chargement du modèle sélectionné
model_path = os.path.join(model_dir, chosen_model_file)
model = load_model(model_path)
print(f"Modèle chargé : {chosen_model_file}")

### 🧪 Préparation et Configuration du Générateur d'Images pour les Données de Test
Ce bloc de code définit le chemin vers le répertoire contenant les images de test et configure un générateur d'images pour préparer ces données. Il spécifie la taille des images et la taille du batch, et applique une normalisation des pixels pour garantir que les images sont prêtes à être évaluées par le modèle. Le générateur est configuré pour une classification binaire, facilitant ainsi l'évaluation des performances du modèle sur des données non vues.

In [ ]:
# Définition du chemin du dossier de test
test_dir = os.path.join(os.getcwd(), 'chest_xray', 'test')

# Paramètres d'entrée des images pour l'évaluation
IMG_SIZE = (224, 224)  # Taille des images d'entrée
BATCH_SIZE = 32  # Taille du batch pour le générateur d'images

# Préparation des images de test avec normalisation des pixels
test_datagen = ImageDataGenerator(rescale=1./255)

# Création du générateur d'images de test
test_generator = test_datagen.flow_from_directory(
    test_dir, # Chemin du dossier de test
    target_size=IMG_SIZE, # Redimensionnement des images
    batch_size=BATCH_SIZE, # Taille du batch
    class_mode='binary'  # Classification binaire (Pneumonie vs Normal)
)

### ⚙️ Évaluation Complète du Modèle sur l'Ensemble de Test
Ce bloc de code effectue une évaluation détaillée du modèle sur les données de test, en calculant diverses métriques de performance telles que la perte, la précision, l'AUC, la précision catégorielle, la précision, et le rappel. Il extrait également les classes réelles et les prédictions du modèle pour générer une matrice de confusion, offrant ainsi une vue d'ensemble des performances du modèle en termes de classification binaire.

In [ ]:
# Évaluation du modèle sur l'ensemble de test
test_loss, test_acc, test_auc, test_categorical_acc, test_precision, test_recall = model.evaluate(test_generator)

# Récupération des classes réelles et des prédictions
y_true = test_generator.classes  # Classes réelles des images de test
y_pred = model.predict(test_generator)  # Prédictions du modèle
y_pred = np.round(y_pred).astype(int)  # Seuil de décision fixé à 0.5 pour classification binaire

# Génération de la matrice de confusion
cm = confusion_matrix(y_true, y_pred)

### 📊 Visualisation des Résultats et Sauvegarde des Logs d'Évaluation
Ce bloc de code génère un identifiant unique pour chaque session d'évaluation basé sur la date et l'heure, puis crée une entrée de log détaillée contenant les métriques de performance du modèle, telles que la précision, la perte, l'AUC, la précision catégorielle, la précision, le rappel, ainsi que la matrice de confusion. Les logs sont sauvegardés dans un fichier JSON pour un suivi ultérieur. En outre, il génère et sauvegarde un graphique de la matrice de confusion pour une visualisation claire des résultats de classification, facilitant ainsi l'analyse des performances du modèle.

In [ ]:
# Génération d'un identifiant de log basé sur la date et l'heure
date = datetime.now().strftime('%Y%m%d%H%M%S')
log_id = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

# Création d'une entrée de log avec les résultats du test
log_entry = {
    log_id: {
        "accuracy": test_acc,
        "loss": test_loss,
        "auc": test_auc,
        "categorical_accuracy": test_categorical_acc,
        "precision": test_precision,
        "recall": test_recall,
        "samples": test_generator.samples,
        "confusion_matrix": cm.tolist(),
        "graph_path": f"reports/graphs/confusion-matrix-{date}.png"
    }
}

# Définition du chemin du fichier de log
log_file_path = "reports/test_log.json"

# Création du répertoire de logs s'il n'existe pas
os.makedirs("reports", exist_ok=True)

# Chargement des logs existants s'ils sont déjà présents
if os.path.exists(log_file_path):
    with open(log_file_path, "r") as f:
        existing_logs = json.load(f)
else:
    existing_logs = {}

# Mise à jour du fichier de log avec la nouvelle entrée
existing_logs.update(log_entry)

# Écriture des logs mis à jour dans le fichier JSON
with open(log_file_path, "w") as f:
    json.dump(existing_logs, f, indent=4)

# Création et affichage de la matrice de confusion avec seaborn
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["NORMAL", "PNEUMONIA"],
    yticklabels=["NORMAL", "PNEUMONIA"]
)
plt.xlabel("Prédictions")
plt.ylabel("Réelles")
plt.title("Matrice de Confusion")

# Création du dossier pour les graphiques s'il n'existe pas
os.makedirs("reports/graphs", exist_ok=True)

# Sauvegarde du graphique de la matrice de confusion
plt.savefig(f"reports/graphs/confusion-matrix-{date}.png")
plt.close()